# Brainrot-V — generate and evaluate

**The only notebook you need.** Run the cells in order.

| Part | Cells | Needs |
| --- | --- | --- |
| **A — generate** | 1–9 | an **A100** |
| **B — evaluate** | 10–18 | a **T4** is enough and costs ~⅛ as much |

Shell steps use `!` so their output streams into the notebook as it happens.
Nothing is hidden behind a captured subprocess.

### If the runtime dies, re-run from cell 1

Colab Pro has no background execution — the session ends when the browser does.

- **Checkpointed.** Finished videos are skipped. VBench is checkpointed per
  *(dimension, video)*; the harness and DINO per video. A crash costs only the
  item in flight.
- **Atomic writes.** A runtime that dies mid-write leaves the previous file
  intact, never half of one.
- **Verified, not trusted.** Finished means the manifest says so *and* the video
  still hashes to what it recorded.
- **`STATUS.md` on Drive** is rewritten as work completes, readable from the
  Drive web UI without opening Colab.

### Three run sets

| `RUN_SET` | Runs | GPU time | Answers |
| --- | ---: | ---: | --- |
| `bridge` | 2 | ~0.8 h | Does Colab reproduce Modal's numbers? **Run first** |
| `pilot` | 8 | ~3.2 h | Does Brainrot-V beat the rolling baseline? |
| `final` | 32 | ~10.8 h | The real comparison, 4 prompts × 4 seeds at 25 s |

An A100 costs roughly **14 compute units per hour**; Colab Pro gives 100 a
month. So bridge ≈ 11 units, pilot ≈ 45, final ≈ 151.

### Why `bridge` first

Every existing result came from an A100-**80**GB with flash-attn. Colab gives an
A100-**40**GB, which takes a different branch at `brainrot_inference.py:205`
(the text encoder is swapped in and out rather than resident), and flash-attn
may not build, in which case attention falls back to another kernel. Neither is
a bug, but Colab clips are **not automatically comparable** with the ones in
`output/llm-methods-30s/`.

`bridge` regenerates `m0` at seeds 1 and 4 so you can check against the Modal
flicker values — **3.21** and **0.86**. Identical runs already differ by
34.7–40.2 dB, so the question is whether the gap sits inside that noise.

### On Drive before you start, at `MyDrive/brainrot-v/`

- `00_setup/models/` — 23 GB. Cell 6 downloads it on the first run; needs
  Google One 100 GB, as Drive's free 15 GB will not hold it.
- `02_reference/` — optional: the 8 existing `m0` clips **and their
  `.manifest.json` files**, under the names they already have. Upload both —
  the manifests carry the provenance that decides whether they may be paired.

Code is cloned from
[the repository](https://github.com/abdanhafidz/BRAINROT-V); nothing else to
upload.

---
# Part A — generate

## 1 · Settings

In [ ]:
# ---- what to run -----------------------------------------------------------
RUN_SET  = "bridge"       # "bridge" | "pilot" | "final"

ROOT     = "/content/drive/MyDrive/brainrot-v"
REPO     = "/content/BRAINROT-V"

REPO_URL = "https://github.com/abdanhafidz/BRAINROT-V"
REPO_REF = "main"         # pin to a commit SHA for an exactly reproducible run

# ---- Part B only -----------------------------------------------------------
DIMENSIONS = [
    "subject_consistency",      # DINO   - identity drift        PRIMARY
    "background_consistency",   # CLIP   - scene stability       PRIMARY
    "motion_smoothness",        # AMT    - temporal artefacts
    "dynamic_degree",           # RAFT   - motion present        CONTROL
    "imaging_quality",          # MUSIQ  - per-frame quality
    "aesthetic_quality",        # LAION  - often needs a manual weights file
]

VBENCH_BATCH   = 4         # videos per VBench call; smaller loses less on a crash
FLICKER_WINDOW = (1, 24)   # seconds averaged for the headline flicker number
DINO_STRIDE    = 2         # frame stride for the decomposition; 1 is exact

print(f"RUN_SET = {RUN_SET}")

## 2 · Mount Drive

Creates the workspace layout if it is not there yet.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!mkdir -p "{ROOT}/00_setup/code" "{ROOT}/00_setup/wheels" "{ROOT}/00_setup/models"
!mkdir -p "{ROOT}/02_reference" "{ROOT}/03_scores" "{ROOT}/04_results/figures"
!mkdir -p "{ROOT}/04_results/tables" "{ROOT}/_cache"
!ls -la "{ROOT}"

## 3 · Check the GPU

Stops here on anything that is not an A100. A T4 would take hours per video and
spend units for an unusable result.

In [ ]:
!nvidia-smi

In [ ]:
gpu_line = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader,nounits
gpu_name, gpu_mib = [x.strip() for x in gpu_line[0].split(",")]
gpu_gb = int(gpu_mib) / 1024

if "A100" not in gpu_name.upper():
    raise SystemExit(f"Got {gpu_name}, not an A100. Recycle the runtime "
                     "(Runtime > Disconnect and delete runtime) and try again.")

# Which branch brainrot_inference.py:205 takes on this card. Modal served 80 GB
# and kept the text encoder resident; a 40 GB card swaps it in and out.
MEMORY_PATH = "resident" if gpu_gb > 60 else "dynamic_swap"

print(f"GPU          {gpu_name}")
print(f"VRAM         {gpu_gb:.0f} GB")
print(f"memory path  {MEMORY_PATH}"
      + ("   <-- differs from every prior run" if MEMORY_PATH != "resident" else ""))

## 4 · Clone the code

The commit SHA goes into every manifest, so a result can always be traced back
to the code that produced it.

In [ ]:
!rm -rf "{REPO}"
!git clone "{REPO_URL}" "{REPO}"
!cd "{REPO}" && git checkout "{REPO_REF}"

In [ ]:
import os, sys
from pathlib import Path

os.chdir(REPO)
sys.path.insert(0, f"{REPO}/notebooks")

commit = !git -C "{REPO}" rev-parse HEAD
COMMIT = commit[0].strip()

import colab_lib as cl
ws = cl.Workspace(ROOT).ensure()

PROMPT_30S = "prompts/brainrot_v_long_horizon_60s.txt"
import hashlib
prompt_sha = hashlib.sha256(Path(PROMPT_30S).read_bytes()).hexdigest()
assert prompt_sha.startswith("e4b595af"), (
    f"Prompt differs from the existing m0 runs (got {prompt_sha[:16]}); the "
    "bridge check would compare different videos")

print(f"commit       {COMMIT}")
print(f"prompt       {prompt_sha[:12]}  matches the eviction-screen runs")
print(f"workspace    {ws.root}")

## 5 · Install packages

`requirements.txt` pins `numpy==1.24.4`, whose wheels stop at **cp311**. Modal
ran Python 3.10 so it never mattered; Colab is newer, pip falls back to building
1.24.4 from source, and that fails because numpy 1.24 still uses `distutils`.

The cell picks the newest 1.x or 2.x release that has a wheel for the running
Python, prints which, and records it in every manifest. The codebase was checked
for numpy-2 removals (`np.float_`, `np.NaN`, `np.product`, …) and uses none.

In [ ]:
import sys

NUMPY_PIN = {(3, 13): "2.2.6",       # 1.26.4 has no cp313 wheel either
             (3, 12): "1.26.4"}.get(sys.version_info[:2], "1.24.4")

print(f"python {sys.version.split()[0]}  ->  numpy=={NUMPY_PIN}")
!sed 's/^numpy==.*/numpy=={NUMPY_PIN}/' "{REPO}/requirements.txt" > /content/requirements-colab.txt
!grep -n "numpy" /content/requirements-colab.txt

In [ ]:
!pip install -r /content/requirements-colab.txt

flash-attn has no prebuilt wheel for this combination and takes 30–60 minutes to
compile, so the build happens **once** and the wheel is cached on Drive. Without
it the code silently falls back to `scaled_dot_product_attention` — which works,
but is a different kernel from every prior run, so it is recorded either way.

In [ ]:
cached = !ls "{ROOT}/00_setup/wheels"/flash_attn-2.7.4*.whl 2>/dev/null
if cached and not cached[0].startswith("ls:"):
    print("installing cached wheel:", cached[0])
    !pip install "{cached[0]}"
else:
    print("no cached wheel - building flash-attn (30-60 min, once) ...")
    !pip wheel flash-attn==2.7.4.post1 --no-build-isolation -w /content/wheels
    !cp /content/wheels/flash_attn-*.whl "{ROOT}/00_setup/wheels/" 2>/dev/null
    !pip install /content/wheels/flash_attn-*.whl 2>/dev/null

In [ ]:
try:
    import flash_attn            # noqa: F401
    FLASH_ATTN = "2.7.4.post1"
except ModuleNotFoundError:
    FLASH_ATTN = None

import torch, numpy
print(f"torch        {torch.__version__}")
print(f"numpy        {numpy.__version__}")
print(f"flash-attn   {FLASH_ATTN or 'ABSENT -> scaled_dot_product_attention'}")
if FLASH_ATTN is None:
    print("             ^ a different attention kernel from every prior run.")
    print("               Recorded per run; such runs must not be pooled.")

## 6 · Models

23.25 GB across four files. Verified on Drive, downloaded if absent, then copied
to session-local SSD — each run is a fresh process that reloads the weights, and
repeated 23 GB reads over Drive cost more in billed GPU time than the copy does.

In [ ]:
WAN  = f"{ROOT}/00_setup/models/wan_models/Wan2.1-T2V-1.3B"
CKPT = f"{ROOT}/00_setup/models/checkpoints/framewise/ar_diffusion.pt"

!du -sh "{ROOT}/00_setup/models" 2>/dev/null || echo "models dir empty"
!ls -la "{WAN}" 2>/dev/null || echo "Wan models not downloaded yet"

In [ ]:
SIZES = {
    f"{WAN}/models_t5_umt5-xxl-enc-bf16.pth":     11_361_920_000,
    f"{WAN}/diffusion_pytorch_model.safetensors":  5_680_000_000,
    f"{WAN}/Wan2.1_VAE.pth":                         510_000_000,
    CKPT:                                          5_680_000_000,
}
ok = lambda p, n: Path(p).exists() and abs(Path(p).stat().st_size - n) <= n * 0.05

missing = [p for p, n in SIZES.items() if not ok(p, n)]
if missing:
    print(f"{len(missing)} file(s) missing - downloading ~23 GB to Drive (once)")
    from huggingface_hub import hf_hub_download, snapshot_download
    if any(p != CKPT for p in missing):
        snapshot_download(repo_id="Wan-AI/Wan2.1-T2V-1.3B", local_dir=WAN)
    if CKPT in missing:
        hf_hub_download(repo_id="zhuhz22/Causal-Forcing",
                        filename="framewise/ar_diffusion.pt",
                        local_dir=f"{ROOT}/00_setup/models/checkpoints")
for p, n in SIZES.items():
    assert ok(p, n), f"{Path(p).name} still missing or truncated"
print("all four model files verified on Drive")

In [ ]:
!mkdir -p /content/models/wan_models /content/models/checkpoints/framewise
!rsync -ah --info=progress2 "{ROOT}/00_setup/models/wan_models/" /content/models/wan_models/
!rsync -ah --info=progress2 "{ROOT}/00_setup/models/checkpoints/" /content/models/checkpoints/
!du -sh /content/models

In [ ]:
CHECKPOINT = "/content/models/checkpoints/framewise/ar_diffusion.pt"

# brainrot_inference.py resolves the Wan assets through repo/wan_models, exactly
# as modal_app.py symlinks them in the container.
!rm -rf "{REPO}/wan_models"
!ln -s /content/models/wan_models "{REPO}/wan_models"
!ls -la "{REPO}/wan_models/"

## 7 · The plan

A run counts as finished only if its manifest says `succeeded` **and** the video
on Drive still hashes to what the manifest recorded — on Modal a stale manifest
was twice mistaken for a live result.

In [ ]:
CONFIG = "configs/llm_methods/brainrot_v_m0-baseline.yaml"


def job(run_id, mode, seed, frames, prompt=PROMPT_30S):
    return dict(run_id=run_id, mode=mode, seed=seed, latent_frames=frames,
                sampling_steps=50, config_path=CONFIG, prompt_file=prompt)


RUN_SETS = {
    # Modal m0 flicker for comparison: seed 1 = 3.21, seed 4 = 0.86
    "bridge": [job(f"bridge-m0-s{s}-30s", "brainrot_v", s, 121) for s in (1, 4)],

    # The baseline arm, pairing against the eight m0 clips that already exist
    "pilot": [job(f"pilot-baseline-s{s}-30s", "baseline", s, 121) for s in range(8)],

    # 4 prompts x 4 seeds x 2 arms at 25 s. The prompts are not written yet, so
    # this set reports them missing until they are.
    "final": [job(f"final-{arm}-p{p}-s{s}-25s", mode, s, 101,
                  f"prompts/eval25/prompt{p}.txt")
              for p in (1, 2, 3, 4) for s in range(4)
              for arm, mode in (("baseline", "baseline"),
                                ("brainrotv", "brainrot_v"))],
}
JOBS = RUN_SETS[RUN_SET]
HOURS_PER = {121: 0.403, 101: 0.337}        # measured 12.0 s per latent frame

print(f"set: {RUN_SET}\n")
missing = sorted({j["prompt_file"] for j in JOBS
                  if not (repo / j["prompt_file"]).exists()})
for name in missing:
    print(f"  MISSING PROMPT  {name}")
if missing:
    print()

TODO = []
for j in JOBS:
    done = ws.run_done(RUN_SET, j["run_id"])
    if not done:
        TODO.append(j)
    print(f"  {'done   ' if done else 'to run '} {j['run_id']}")

hours = sum(HOURS_PER[j["latent_frames"]] for j in TODO)
print(f"\n{len(TODO)} of {len(JOBS)} left   ~{hours:.1f} GPU-hours   "
      f"~{hours * 14:.0f} units")
print(f"\nstatus written to {ws.write_status()}")

## 8 · Generate

The only cell that costs GPU time. Each video takes 20–24 minutes and streams
its output here as it runs. The manifest is written even on a crash or
interrupt, and `STATUS.md` is refreshed after each one. Safe to stop and re-run.

In [ ]:
import time, shutil, json

if missing_prompts:
    raise SystemExit("Write the missing prompt files first (see cell 7).")
if not TODO:
    raise SystemExit("Nothing left to run.")

for n, j in enumerate(TODO, 1):
    run_id = j["run_id"]
    print(f"\n{'=' * 70}\n[{n}/{len(TODO)}] {run_id}\n{'=' * 70}")

    # Colab can reassign the runtime between runs; never mix hardware in a set.
    now = !nvidia-smi --query-gpu=name --format=csv,noheader
    assert now[0].strip() == gpu_name, f"GPU changed: {gpu_name} -> {now[0]}"

    out_dir = f"/content/out/{run_id}"
    out_mp4 = f"{out_dir}/video.mp4"
    log     = str(ws.log(RUN_SET, run_id))
    !mkdir -p "{out_dir}"

    cmd = (f'python -u brainrot_inference.py'
           f' --prompt_file "{j["prompt_file"]}"'
           f' --checkpoint_path "{CHECKPOINT}"'
           f' --output_file "{out_mp4}"'
           f' --num_output_frames {j["latent_frames"]}'
           f' --sampling_steps {j["sampling_steps"]}'
           f' --seed {j["seed"]}'
           f' --mode {j["mode"]}'
           f' --config_path "{j["config_path"]}"')

    record = dict(
        j, run_set=RUN_SET, platform="colab", commit=COMMIT, repo=REPO_URL,
        prompt_sha256=hashlib.sha256(Path(j["prompt_file"]).read_bytes()).hexdigest(),
        config=Path(j["config_path"]).read_text(),
        command=cmd,
        gpu=gpu_name, gpu_total_gb=round(gpu_gb, 1),
        memory_path=MEMORY_PATH,        # resident vs dynamic_swap
        flash_attn=FLASH_ATTN,          # None means the SDPA fallback
        torch_version=torch.__version__, numpy_version=numpy.__version__,
        started_at=cl.utcnow(), status="running")
    cl.write_json(ws.manifest(RUN_SET, run_id), record)

    torch.cuda.reset_peak_memory_stats()
    began = time.monotonic()
    try:
        # tee keeps a copy on Drive while the output streams into the notebook.
        !{cmd} 2>&1 | tee "{log}"
        record["exit_code"] = _exit_code

        if Path(out_mp4).exists():
            final = str(ws.video(RUN_SET, run_id))
            !cp "{out_mp4}" "{final}"
            sidecar = out_mp4.replace(".mp4", ".schedule.json")
            if Path(sidecar).exists():
                !cp "{sidecar}" "{ws.schedule(RUN_SET, run_id)}"
            record.update(status="succeeded", output_path=final,
                          output_bytes=Path(final).stat().st_size,
                          output_sha256=cl.sha256_file(final))
        else:
            record["status"] = "failed"
    except KeyboardInterrupt:
        record["status"] = "interrupted"
        raise
    finally:
        record["wall_seconds"] = round(time.monotonic() - began, 1)
        record["peak_vram_gb"] = round(torch.cuda.max_memory_allocated() / 2**30, 2)
        record["finished_at"] = cl.utcnow()
        cl.write_json(ws.manifest(RUN_SET, run_id), record)
        ws.write_status()              # Drive shows progress immediately

    print(f"\n-> {record['status']}  {record['wall_seconds']:.0f}s  "
          f"peak {record['peak_vram_gb']} GB")

print(f"\ndone - {ws.root}/STATUS.md")

## 9 · Check

In [ ]:
rows = ws.inventory(RUN_SET)
if not rows:
    print("nothing generated yet")
else:
    width = max(len(r["run_id"]) for r in rows)
    print(f"{'run':<{width}}  {'ok':<4} {'wall':>7} {'path':<14} attention")
    for r in rows:
        print(f"{r['run_id']:<{width}}  {'yes' if r['verified'] else 'NO ':<4} "
              f"{(r['wall_seconds'] or 0):>6.0f}s {str(r['memory_path']):<14} "
              f"{r['flash_attn'] or 'sdpa'}")

    paths = {r["memory_path"] for r in rows if r["verified"]}
    kernels = {r["flash_attn"] for r in rows if r["verified"]}
    if len(paths) > 1 or len(kernels) > 1:
        print(f"\nWARNING - mixed execution paths {paths} / {kernels}. Runs made "
              "on different hardware or attention kernels are not comparable.")

    bad = [r["run_id"] for r in rows if not r["verified"]]
    print(f"\n{len(rows) - len(bad)} of {len(rows)} verified"
          + (f"; re-run cell 4 for: {', '.join(bad)}" if bad else ""))

ws.write_status()
print(f"\nStatus table: {ws.root / 'STATUS.md'}")
print(f"Videos:       {ws.runs / RUN_SET / 'video'}")

In [ ]:
!cat "{ROOT}/STATUS.md"

---
# Part B — evaluate

Scores the videos and writes every figure and table to `04_results/`.

**Switch the runtime to a T4 first** if you want to save roughly 25 units —
nothing below needs an A100. After switching, run **cell 1**, then **cell 2**,
then come straight here. Skip cells 3–8: cell 3 asserts an A100 and will stop.

## 10 · Install VBench

In [ ]:
!pip install vbench matplotlib pandas scipy

In [ ]:
import os, sys, json, re, shutil, warnings
from pathlib import Path

# A fresh runtime after a T4 switch has no repo.
if not Path(REPO).exists():
    !git clone "{REPO_URL}" "{REPO}"
    !cd "{REPO}" && git checkout "{REPO_REF}"
os.chdir(REPO)
sys.path.insert(0, f"{REPO}/notebooks")
import colab_lib as cl
ws = cl.Workspace(ROOT).ensure()

# VBench's constructor demands this path but never reads it in custom_input mode.
FULL_INFO = "/content/VBench_full_info.json"
!test -f "{FULL_INFO}" || wget -q -O "{FULL_INFO}" https://raw.githubusercontent.com/Vchitect/VBench/master/vbench/VBench_full_info.json || echo "[]" > "{FULL_INFO}"

# Weights are large; keeping them on Drive avoids refetching every session.
os.environ["VBENCH_CACHE_DIR"] = f"{ROOT}/_cache/vbench_weights"
!mkdir -p "{ROOT}/_cache/vbench_weights"

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE,
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 11 · Find and pair the videos

Reads `01_runs/<set>/video/` plus anything in `02_reference/`. Pairs form on
**(prompt, seed, duration)**, so each comparison holds everything constant
except the arm.

In [ ]:
import pandas as pd

SETS = cl.RUN_SETS if RUN_SET == "all" else (RUN_SET,)
SOURCES = [(ws.runs / s / "video", s) for s in SETS]
# Pairing a pilot needs its partner arm, which lives in 02_reference/ (clips
# generated elsewhere) or came from the bridge set.
SOURCES += [(ws.reference, "reference"), (ws.runs / "bridge" / "video", "bridge")]

# run_id -> (arm, prompt key, seed, seconds). Covers the runner's three sets and
# the original Modal names, so Modal clips can be dropped into reference/ as-is.
PATTERNS = [
    (r"^bridge-m0-s(\d+)-(\d+)s$",               lambda m: ("brainrotv", "p0", int(m[1]), int(m[2]))),
    (r"^pilot-baseline-s(\d+)-(\d+)s$",          lambda m: ("baseline",  "p0", int(m[1]), int(m[2]))),
    (r"^final-(\w+)-p(\d+)-s(\d+)-(\d+)s$",      lambda m: (m[1], f"p{m[2]}", int(m[3]), int(m[4]))),
    (r"^llm-m0-baseline-s(\d+)-(\d+)s$",         lambda m: ("brainrotv", "p0", int(m[1]), int(m[2]))),
]


def classify(stem):
    for pattern, build in PATTERNS:
        match = re.match(pattern, stem)
        if match:
            return build(match)
    return None


records, seen = [], set()
for folder, origin in SOURCES:
    if not folder.exists():
        continue
    for video in sorted(folder.glob("*.mp4")):
        if video.stem in seen:
            continue
        parsed = classify(video.stem)
        if not parsed:
            print(f"  skipped (unrecognised name): {video.name}")
            continue
        seen.add(video.stem)
        arm, prompt, seed, seconds = parsed

        # Provenance decides whether two clips may be compared at all, so an
        # unknown value must stay visible rather than becoming None - a None is
        # dropped by the mixed-path check below and would hide the difference.
        meta = {}
        for candidate in cl.RUN_SETS:
            meta = cl.read_json(ws.manifest(candidate, video.stem), {}) or meta
            if meta:
                break
        if not meta:
            # A clip generated elsewhere: look for its manifest beside it, under
            # either the Modal name or the plain one.
            for sidecar in (video.with_suffix(".manifest.json"),
                            video.with_suffix(".json")):
                meta = cl.read_json(sidecar, {}) or meta
                if meta:
                    break
        meta.setdefault("memory_path", "unknown")
        meta.setdefault("flash_attn", "unknown")
        records.append(dict(
            run_id=video.stem, path=str(video), origin=origin, arm=arm,
            prompt=prompt, seed=seed, seconds=seconds,
            memory_path=meta.get("memory_path"), flash_attn=meta.get("flash_attn"),
            wall_seconds=meta.get("wall_seconds"), peak_vram_gb=meta.get("peak_vram_gb"),
        ))

videos = pd.DataFrame(records)
if videos.empty:
    raise SystemExit("No videos found. Run colab_runner.ipynb first.")

# Pair: one row per (prompt, seed, seconds) that has both arms.
wide = videos.pivot_table(index=["prompt", "seed", "seconds"], columns="arm",
                          values="run_id", aggfunc="first")
PAIRS = wide.dropna(subset=[c for c in ("baseline", "brainrotv") if c in wide]).reset_index() \
    if {"baseline", "brainrotv"} <= set(wide.columns) else pd.DataFrame()

print(f"{len(videos)} videos, {len(PAIRS)} complete pairs\n")
print(videos[["run_id", "origin", "arm", "prompt", "seed", "memory_path",
              "flash_attn"]].to_string(index=False))

paths = set(videos.memory_path.dropna()); kernels = set(videos.flash_attn.dropna())
if len(paths) > 1 or len(kernels) > 1:
    print(f"\n*** MIXED EXECUTION PATHS: memory_path={paths}, flash_attn={kernels}")
    print("    Clips made on different hardware or with a different attention")
    print("    kernel are not directly comparable. Either upload the missing")
    print("    manifests beside the clips in 02_reference/, or run the bridge")
    print("    check to show the difference sits inside run-to-run noise.")
    print(videos.groupby(["origin", "memory_path", "flash_attn"])
          .size().rename("videos").to_string())

if PAIRS.empty:
    print("\nNo complete pairs yet - scoring still works, comparisons will be skipped.")
else:
    print(f"\npairs:\n{PAIRS.to_string(index=False)}")

## 12 · Local harness

In [ ]:
harness = cl.Checkpoint(ws.score(f"harness-{RUN_SET}"))
todo = [r for r in videos.itertuples() if not harness.done(r.run_id)]
print(f"{len(harness)} cached, {len(todo)} to measure")

for start in range(0, len(todo), VBENCH_BATCH):
    batch = todo[start:start + VBENCH_BATCH]
    paths = " ".join(f'"{r.path}"' for r in batch)
    print(f"\n--- {start + 1}-{start + len(batch)} of {len(todo)}")
    !python video_metrics.py /content/harness.json {paths}
    by_name = {Path(e["file"]).stem: e
               for e in json.loads(Path("/content/harness.json").read_text())}
    harness.update({r.run_id: by_name[Path(r.path).stem]
                    for r in batch if Path(r.path).stem in by_name})

scored = harness.data
print(f"\nharness: {len(scored)} video(s)  ->  {harness.path.name}")


def harness_row(run_id):
    entry = scored.get(run_id)
    if not entry:
        return {}
    lo, hi = FLICKER_WINDOW
    out = {"clean_horizon": entry.get("clean_horizon_seconds")}
    for metric in ("flicker", "speckle", "streak"):
        series = entry["metrics"][metric]["per_second"][lo:hi]
        out[metric] = sum(series) / len(series) if series else None
    return out

## 13 · VBench

Each dimension is scored separately and checkpointed per *(dimension, video)*.
One that fails — `aesthetic_quality` needs a weights file VBench does not
download — is reported and skipped; the rest still complete.

In [ ]:
from vbench import VBench

# Checkpointed per (dimension, video), so a session that dies part-way through a
# dimension keeps the videos it already scored rather than repeating them.
scores = cl.Checkpoint(ws.score(f"vbench-{RUN_SET}"))
key = lambda dimension, run_id: f"{dimension}|{run_id}"
failed = {}

for dimension in DIMENSIONS:
    pending = [r for r in videos.itertuples()
               if not scores.done(key(dimension, r.run_id))]
    have = len(videos) - len(pending)
    if not pending:
        print(f"{dimension:<24} cached ({have} videos)")
        continue

    print(f"{dimension:<24} {have} cached, {len(pending)} to score", flush=True)
    for start in range(0, len(pending), VBENCH_BATCH):
        batch = pending[start:start + VBENCH_BATCH]
        stage = Path(f"/content/vb_stage/{dimension}")
        if stage.exists():
            shutil.rmtree(stage)
        stage.mkdir(parents=True)
        for row in batch:
            shutil.copy2(row.path, stage / f"{row.run_id}.mp4")

        # Folder mode asserts every video has a prompt. None of the six
        # custom-input dimensions read it - they are all reference-free - but
        # the entry must exist.
        prompts = {f"{r.run_id}.mp4": "a fox in a forest" for r in batch}
        out = Path(f"/content/vb_out/{dimension}")
        try:
            VBench(DEVICE, str(FULL_INFO), str(out)).evaluate(
                videos_path=str(stage), name=dimension, prompt_list=prompts,
                dimension_list=[dimension], mode="custom_input")
            payload = json.loads((out / f"{dimension}_eval_results.json").read_text())
            # {dimension: [aggregate, [{video_path, video_results}, ...]]}
            _, per_video = payload[dimension]
            scores.update({key(dimension, Path(v["video_path"]).stem): v["video_results"]
                           for v in per_video})
            print(f"{' ':<24} batch {start // VBENCH_BATCH + 1}: "
                  f"{len(per_video)} scored", flush=True)
        except Exception as error:
            failed[dimension] = f"{type(error).__name__}: {error}"
            print(f"{dimension:<24} FAILED - {failed[dimension]}")
            print(f"{' ':<24} skipping; it will show as '-' in the tables")
            break

# Regroup the flat checkpoint into {dimension: {run_id: score}}.
vbench = {}
for flat, value in scores.data.items():
    dimension, run_id = flat.split("|", 1)
    vbench.setdefault(dimension, {})[run_id] = value

print(f"\n{len(vbench)} of {len(DIMENSIONS)} dimensions have scores "
      f"({len(scores)} video-dimension pairs cached)")
for dimension, reason in failed.items():
    print(f"  FAILED  {dimension}: {reason}")

## 14 · Long-range vs short-range drift

In [ ]:
import numpy as np

store = cl.Checkpoint(ws.score(f"dino-{RUN_SET}"))
dino = store.data
pending = [r for r in videos.itertuples() if not store.done(r.run_id)]

if pending:
    import cv2
    import torch.nn.functional as F
    from torchvision import transforms

    model = torch.hub.load("facebookresearch/dino:main", "dino_vitb16").to(DEVICE).eval()
    prep = transforms.Compose([
        transforms.ToTensor(),
        transforms.Resize(224, antialias=True),
        transforms.CenterCrop(224),
        transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
    ])

    @torch.no_grad()
    def decompose(path):
        capture = cv2.VideoCapture(path)
        first = previous = None
        to_first, to_prev = [], []
        index = 0
        while True:
            ok, frame = capture.read()
            if not ok:
                break
            if index % DINO_STRIDE:
                index += 1
                continue
            index += 1
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            feature = F.normalize(model(prep(rgb).unsqueeze(0).to(DEVICE)), dim=-1)
            if first is None:
                first = feature
            else:
                to_first.append(max(0.0, F.cosine_similarity(first, feature).item()))
                to_prev.append(max(0.0, F.cosine_similarity(previous, feature).item()))
            previous = feature
        capture.release()
        return {"sim_to_first": float(np.mean(to_first)),
                "sim_to_prev": float(np.mean(to_prev)),
                "composite": float(np.mean(to_first) + np.mean(to_prev)) / 2,
                "to_first_series": [round(x, 5) for x in to_first]}

    for n, row in enumerate(pending, 1):
        print(f"  [{n}/{len(pending)}] {row.run_id}", flush=True)
        store.put(row.run_id, decompose(row.path))   # saved after each video

print(f"decomposition: {len(dino)} video(s)")
for run_id, value in sorted(dino.items()):
    print(f"  {run_id:<34} first {value['sim_to_first']:.4f}   "
          f"prev {value['sim_to_prev']:.4f}   composite {value['composite']:.4f}")

## 15 · One table of everything

In [ ]:
rows = []
for row in videos.itertuples():
    entry = dict(run_id=row.run_id, arm=row.arm, prompt=row.prompt, seed=row.seed,
                 seconds=row.seconds, origin=row.origin,
                 wall_seconds=row.wall_seconds, peak_vram_gb=row.peak_vram_gb)
    entry.update(harness_row(row.run_id))
    for dimension in DIMENSIONS:
        entry[dimension] = vbench.get(dimension, {}).get(row.run_id)
    decomposed = dino.get(row.run_id, {})
    entry["sim_to_first"] = decomposed.get("sim_to_first")
    entry["sim_to_prev"] = decomposed.get("sim_to_prev")
    rows.append(entry)

DATA = pd.DataFrame(rows).sort_values(["prompt", "seed", "arm"]).reset_index(drop=True)
DATA.to_csv(ws.table(f"per-video-{RUN_SET}.csv"), index=False)

METRICS = [d for d in DIMENSIONS if d in vbench] + \
          ["sim_to_first", "sim_to_prev", "flicker", "speckle", "streak", "clean_horizon"]
HIGHER_IS_BETTER = {*DIMENSIONS, "sim_to_first", "sim_to_prev", "clean_horizon"}

pd.set_option("display.width", 200, "display.max_columns", 40)
print(DATA[["run_id", "arm", "seed", *[m for m in METRICS if m in DATA]]]
      .to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print(f"\nwrote {ws.table(f'per-video-{RUN_SET}.csv')}")

## 16 · Statistics

In [ ]:
from math import comb
from scipy import stats


def sign_test(wins, n):
    if n == 0:
        return float("nan")
    lo = min(wins, n - wins)
    return min(1.0, 2 * sum(comb(n, i) for i in range(lo + 1)) / 2 ** n)


def paired(metric):
    """One row per pair: baseline value, brainrot-v value, delta, win flag."""
    out = []
    for pair in PAIRS.itertuples():
        base = DATA[DATA.run_id.eq(pair.baseline)][metric]
        prop = DATA[DATA.run_id.eq(pair.brainrotv)][metric]
        if base.empty or prop.empty or pd.isna(base.iloc[0]) or pd.isna(prop.iloc[0]):
            continue
        b, p = float(base.iloc[0]), float(prop.iloc[0])
        better = p > b if metric in HIGHER_IS_BETTER else p < b
        out.append(dict(prompt=pair.prompt, seed=pair.seed, baseline=b,
                        brainrotv=p, delta=p - b,
                        ratio=(p / b if b else float("nan")),
                        tie=(p == b), win=bool(better)))
    return pd.DataFrame(out)


def live_pairs(table):
    """Exact ties carry no sign, so a sign test must drop them rather than count
    them as losses - otherwise an all-tied metric reports a spurious p = 0.008."""
    return table[~table.tie]


SUMMARY, PAIRED = [], {}
if PAIRS.empty:
    print("No complete pairs - nothing to test yet.")
else:
    for metric in METRICS:
        table = paired(metric)
        if table.empty:
            continue
        PAIRED[metric] = table
        live = live_pairs(table)
        n, wins, ties = len(live), int(live.win.sum()), int(table.tie.sum())
        p_sign = sign_test(wins, n) if n else float("nan")
        try:
            p_wilcoxon = (stats.wilcoxon(live.baseline, live.brainrotv).pvalue
                          if n else float("nan"))
        except ValueError:
            p_wilcoxon = float("nan")
        SUMMARY.append(dict(
            metric=metric, n=n, ties=ties, wins=wins,
            win_rate=(wins / n if n else float("nan")),
            baseline=table.baseline.mean(), brainrotv=table.brainrotv.mean(),
            delta=table.delta.mean(), median_delta=table.delta.median(),
            p_sign=p_sign, p_wilcoxon=p_wilcoxon))

    SUMMARY = pd.DataFrame(SUMMARY)
    print(SUMMARY.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

    primary = "subject_consistency"
    if primary in PAIRED:
        table = live_pairs(PAIRED[primary])
        n, wins = len(table), int(table.win.sum())
        need = next((k for k in range(n, n // 2, -1) if sign_test(k, n) < 0.05), None)
        print(f"\nPRIMARY - {primary}")
        print(f"  {wins} of {n} paired wins, two-sided sign test p = {sign_test(wins, n):.4f}")
        print(f"  minimum for p < 0.05 at n = {n} is {need}/{n}"
              if need else "  no win count reaches p < 0.05 at this n")
        print(f"  mean delta {table.delta.mean():+.4f}   "
              f"median {table.delta.median():+.4f}   "
              f"range {table.delta.min():+.4f} to {table.delta.max():+.4f}")

        # Does the effect depend on how hard the seed was? This is the structure
        # that was significant (rho = -0.81) on the eviction data while the
        # average effect was not.
        if len(table) >= 4:
            rho, p_rho = stats.spearmanr(table.baseline, table.delta)
            print(f"  Spearman rho(baseline, delta) = {rho:+.3f}, p = {p_rho:.4f}"
                  + ("   <- conditional effect" if p_rho < 0.05 else ""))

    control = "dynamic_degree"
    if control in PAIRED:
        table = live_pairs(PAIRED[control])
        n = len(table)
        reduced = int((~table.win).sum())            # pairs with LESS motion
        p_ctrl = sign_test(reduced, n) if n else float("nan")
        print(f"\nCONTROL - {control}")
        print(f"  motion reduced in {reduced} of {n} pairs, "
              f"mean delta {table.delta.mean():+.4f}, sign test p = {p_ctrl:.4f}")
        if reduced > n / 2 and p_ctrl < 0.05:
            print("  *** Motion is significantly REDUCED. A consistency win alongside")
            print("      this means the method stabilises by freezing - report it as")
            print("      a trade, not a win.")
        else:
            print("  OK - no significant loss of motion, so any consistency gain is")
            print("     not explained by the video simply moving less.")

## 17 · Figures

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

BASE_C, PROP_C, GRID = "#b4453c", "#2f6f9f", "#d8d8d8"
FIGS = ws.figures


def tidy(ax):
    ax.grid(axis="y", color=GRID, linewidth=0.6)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)


def save(fig, name):
    path = FIGS / f"{name}-{RUN_SET}.png"
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print("  ", path.name)
    return path


print("figures written to", FIGS)

# --- 1. paired lines, primary metric ---------------------------------------
if "subject_consistency" in PAIRED:
    table = PAIRED["subject_consistency"].sort_values(["prompt", "seed"])
    labels = [f"{r.prompt}/s{r.seed}" for r in table.itertuples()]
    x = range(len(table))
    fig, ax = plt.subplots(figsize=(max(7, len(table) * 0.8), 4.2))
    ax.plot(x, table.baseline, "o-", color=BASE_C, label="baseline (rolling)", lw=2)
    ax.plot(x, table.brainrotv, "s-", color=PROP_C, label="Brainrot-V", lw=2)
    for i, row in enumerate(table.itertuples()):
        ax.plot([i, i], [row.baseline, row.brainrotv], color="#999", lw=1, zorder=0)
    wins, n = int(table.win.sum()), len(table)
    ax.set_xticks(list(x)); ax.set_xticklabels(labels, fontsize=8, rotation=45, ha="right")
    ax.set_ylabel("subject consistency (higher better)", fontsize=9)
    ax.set_title(f"Paired by seed: {wins} of {n} wins, p = {sign_test(wins, n):.3f}",
                 fontsize=10)
    ax.legend(fontsize=9, frameon=False); tidy(ax)
    save(fig, "paired-subject-consistency")

# --- 2. effect against difficulty ------------------------------------------
if "subject_consistency" in PAIRED and len(PAIRED["subject_consistency"]) >= 4:
    table = PAIRED["subject_consistency"]
    fig, ax = plt.subplots(figsize=(5.5, 4.2))
    ax.scatter(table.baseline, table.delta,
               c=[PROP_C if w else BASE_C for w in table.win], s=70, zorder=3)
    for row in table.itertuples():
        ax.annotate(f"s{row.seed}", (row.baseline, row.delta),
                    textcoords="offset points", xytext=(7, 3), fontsize=8)
    ax.axhline(0, color="#888", lw=1, ls="--")
    rho, p_rho = stats.spearmanr(table.baseline, table.delta)
    ax.set_xlabel("baseline score (lower = harder seed)", fontsize=9)
    ax.set_ylabel("delta (Brainrot-V - baseline)", fontsize=9)
    ax.set_title(f"Effect vs difficulty: rho = {rho:+.2f}, p = {p_rho:.3f}", fontsize=10)
    tidy(ax)
    save(fig, "effect-vs-difficulty")

# --- 3. every dimension, mean with per-pair points -------------------------
if SUMMARY is not None and len(SUMMARY):
    dims = [m for m in SUMMARY.metric if m in DIMENSIONS]
    if dims:
        fig, axes = plt.subplots(1, len(dims), figsize=(2.3 * len(dims), 4.2),
                                 squeeze=False)
        for ax, metric in zip(axes[0], dims):
            table = PAIRED[metric]
            # A bar from zero would hide the effect: these scores all sit near
            # 0.9 and the difference lives in the third decimal. One line per
            # pair, on a y-range fitted to the data.
            for row in table.itertuples():
                ax.plot([0, 1], [row.baseline, row.brainrotv],
                        color=(PROP_C if row.win else BASE_C),
                        lw=1.1, marker="o", ms=4, alpha=0.65, zorder=2)
            ax.plot([0, 1], [table.baseline.mean(), table.brainrotv.mean()],
                    color="#222", lw=2.4, marker="D", ms=6, zorder=4, label="mean")
            values = list(table.baseline) + list(table.brainrotv)
            pad = (max(values) - min(values) or 0.01) * 0.12
            ax.set_ylim(min(values) - pad, max(values) + pad)
            live = live_pairs(table)
            wins, n = int(live.win.sum()), len(live)
            ax.set_xlim(-0.3, 1.3)
            ax.set_xticks([0, 1]); ax.set_xticklabels(["base", "B-V"], fontsize=9)
            ax.tick_params(axis="y", labelsize=7)
            ax.set_title(f"{metric.replace('_', chr(10))}\n{wins}/{n}  "
                         f"p={sign_test(wins, n):.3f}" if n else metric, fontsize=8)
            tidy(ax)
        axes[0][0].legend(fontsize=8, frameon=False)
        save(fig, "all-dimensions")

# --- 4. the decomposition ---------------------------------------------------
if "sim_to_first" in PAIRED and "sim_to_prev" in PAIRED:
    fig, ax = plt.subplots(figsize=(6, 4.2))
    parts = ["sim_to_first", "sim_to_prev"]
    offsets = [-0.17, 0.17]
    for shift, arm, colour in ((o, a, c) for o, (a, c) in
                               zip(offsets, (("baseline", BASE_C), ("brainrotv", PROP_C)))):
        ax.bar([i + shift for i in range(len(parts))],
               [PAIRED[p][arm].mean() for p in parts], width=0.33,
               color=colour, label=arm, zorder=2)
    for i, part in enumerate(parts):                  # put the gap on the chart
        t = PAIRED[part]
        top = max(t.baseline.mean(), t.brainrotv.mean())
        ax.annotate(f"{t.delta.mean():+.4f}", (i, top), ha="center",
                    textcoords="offset points", xytext=(0, 6), fontsize=10,
                    fontweight="bold",
                    color=PROP_C if t.delta.mean() > 0 else BASE_C)
    ax.set_ylim(0, max(PAIRED[p].brainrotv.mean() for p in parts) * 1.18)
    ax.set_xticks(range(len(parts)))
    ax.set_xticklabels(["sim to FIRST frame\n(long-range - the claim)",
                        "sim to PREVIOUS frame\n(short-range)"], fontsize=9)
    ax.set_ylabel("DINO cosine similarity", fontsize=9)
    ax.set_title("The official metric averages these two", fontsize=10)
    ax.legend(fontsize=9, frameon=False, loc="lower right"); tidy(ax)
    save(fig, "decomposition")

# --- 5. drift over time -----------------------------------------------------
series = {r.run_id: dino[r.run_id]["to_first_series"]
          for r in videos.itertuples() if r.run_id in dino}
if series:
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    for row in videos.itertuples():
        values = series.get(row.run_id)
        if not values:
            continue
        seconds = np.arange(len(values)) * DINO_STRIDE / 16
        ax.plot(seconds, values, lw=1.2, alpha=0.75,
                color=PROP_C if row.arm == "brainrotv" else BASE_C)
    ax.plot([], [], color=BASE_C, label="baseline")
    ax.plot([], [], color=PROP_C, label="Brainrot-V")
    ax.set_xlabel("seconds", fontsize=9)
    ax.set_ylabel("similarity to first frame", fontsize=9)
    ax.set_title("Identity drift over the rollout", fontsize=10)
    ax.legend(fontsize=9, frameon=False); tidy(ax)
    save(fig, "drift-over-time")

# --- 6. flicker over time ---------------------------------------------------
if scored:
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    for row in videos.itertuples():
        entry = scored.get(row.run_id)
        if not entry:
            continue
        values = entry["metrics"]["flicker"]["per_second"]
        ax.plot(range(len(values)), values, lw=1.2, alpha=0.75,
                color=PROP_C if row.arm == "brainrotv" else BASE_C)
    ax.plot([], [], color=BASE_C, label="baseline")
    ax.plot([], [], color=PROP_C, label="Brainrot-V")
    ax.set_xlabel("seconds", fontsize=9)
    ax.set_ylabel("flicker (lower better)", fontsize=9)
    ax.set_title("Flicker across the rollout", fontsize=10)
    ax.legend(fontsize=9, frameon=False); tidy(ax)
    save(fig, "flicker-over-time")

## 18 · Tables

In [ ]:
def fmt(value, places=4):
    return "—" if value is None or (isinstance(value, float) and pd.isna(value)) \
        else f"{value:.{places}f}"


lines = [f"# Brainrot-V evaluation — `{RUN_SET}`", ""]
lines += [f"{len(videos)} videos, {len(PAIRS)} paired comparisons.",
          f"Dimensions scored: {', '.join(vbench) or 'none'}.", ""]

# Table 1 - headline
lines += ["## Table 1 — Headline", "",
          "| Metric | Baseline | Brainrot-V | Δ | Paired wins | Sign test p |",
          "| --- | ---: | ---: | ---: | :---: | ---: |"]
for row in (SUMMARY.itertuples() if len(SUMMARY) else []):
    flag = " **(control)**" if row.metric == "dynamic_degree" else ""
    ties = f" (+{row.ties} tied)" if row.ties else ""
    lines.append(f"| `{row.metric}`{flag} | {fmt(row.baseline)} | {fmt(row.brainrotv)} "
                 f"| {row.delta:+.4f} | {row.wins}/{row.n}{ties} | {fmt(row.p_sign, 4)} |")

# Table 2 - decomposition
lines += ["", "## Table 2 — subject_consistency decomposed", "",
          "| Component | Baseline | Brainrot-V | Δ | Note |",
          "| --- | ---: | ---: | ---: | --- |"]
for part, note in (("sim_to_first", "long-range anchor — the claim"),
                   ("sim_to_prev", "short-range; both arms keep recent frames")):
    if part in PAIRED:
        t = PAIRED[part]
        lines.append(f"| `{part}` | {fmt(t.baseline.mean())} | {fmt(t.brainrotv.mean())} "
                     f"| {t.delta.mean():+.4f} | {note} |")

# Table 3 - per pair
lines += ["", "## Table 3 — Every pair", "",
          "| Prompt | Seed | Baseline | Brainrot-V | Δ | Win |",
          "| --- | ---: | ---: | ---: | ---: | :---: |"]
for row in (PAIRED.get("subject_consistency", pd.DataFrame()).itertuples()):
    lines.append(f"| {row.prompt} | {row.seed} | {fmt(row.baseline)} | "
                 f"{fmt(row.brainrotv)} | {row.delta:+.4f} | "
                 f"{'yes' if row.win else '**no**'} |")

# Table 4 - local harness
lines += ["", "## Table 4 — Local harness", "",
          "| Metric | Baseline | Brainrot-V | Δ | Paired wins |",
          "| --- | ---: | ---: | ---: | :---: |"]
for metric in ("flicker", "speckle", "streak", "clean_horizon"):
    if metric in PAIRED:
        t = PAIRED[metric]
        lines.append(f"| {metric} | {fmt(t.baseline.mean())} | {fmt(t.brainrotv.mean())} "
                     f"| {t.delta.mean():+.4f} | {int(t.win.sum())}/{len(t)} |")

# Table 5 - efficiency and provenance
lines += ["", "## Table 5 — Efficiency and provenance", "",
          "| Arm | Videos | Mean wall (s) | Peak VRAM (GB) | Memory path | Attention |",
          "| --- | ---: | ---: | ---: | --- | --- |"]
for arm, group in videos.groupby("arm"):
    lines.append(f"| {arm} | {len(group)} | {fmt(group.wall_seconds.mean(), 0)} | "
                 f"{fmt(group.peak_vram_gb.mean(), 1)} | "
                 f"{'/'.join(sorted(set(group.memory_path.dropna()))) or '—'} | "
                 f"{'/'.join(sorted(set(group.flash_attn.dropna()))) or 'sdpa'} |")

lines += ["", "## Active KV tokens", "",
          "| | Baseline | Brainrot-V | Δ |", "| --- | ---: | ---: | ---: |",
          "| Active KV tokens (21-frame eval) | 32,760 | 22,328 | **−31.8%** |"]

report = ws.table(f"report-{RUN_SET}.md")
cl.write_atomic(report, "\n".join(lines))
print("\n".join(lines))
ws.write_status()
print(f"\n\nwrote   {report}")
print(f"figures {FIGS}")
print(f"status  {ws.root / 'STATUS.md'}")

In [ ]:
!ls -la "{ROOT}/04_results/figures" "{ROOT}/04_results/tables"
!cat "{ROOT}/04_results/tables/report-{RUN_SET}.md"

---
## Done

| Where | What |
| --- | --- |
| `STATUS.md` | which runs finished, on what hardware |
| `01_runs/<set>/video/` | the videos |
| `01_runs/<set>/manifest/` | run records with SHA-256 and commit, for `reproducibility.mdx` |
| `03_scores/` | cached metrics — delete a file to force a rescore |
| `04_results/figures/` | six PNGs |
| `04_results/tables/` | per-video CSV and `report-<set>.md`, ready to paste |

**After `bridge`:** compare the two flicker numbers against Modal's **3.21** and
**0.86**. Inside the 34.7–40.2 dB noise means the eight existing `m0` clips are
reusable — put them in `02_reference/` with their manifests and set
`RUN_SET = "pilot"`. Outside it means both arms must be regenerated here.